# TensiMenu Model 02 — Weighted Cosine + MinMaxScaler

**Author**: Whenny Zenica

**Pendekatan**:
- Normalisasi: **MinMaxScaler** (rentang [0,1]) — alternatif dari StandardScaler
- Similarity: **Weighted Cosine Similarity** — bobot fitur diterapkan pada vektor sebelum cosine
- DASH Score: tetap formula yang sama untuk fair comparison

**Hipotesis**: MinMaxScaler lebih cocok untuk fitur dengan distribusi non-Gaussian (kebanyakan nutrisi punya skewness tinggi). Bobot eksplisit memastikan natrium dan kalium punya pengaruh lebih besar pada similarity.

---

In [ ]:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
try:
    from google.colab import files, drive
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_BASE = '/content/drive/MyDrive/TensiMenu_ML'
    print(f'Google Drive mounted. Base: {DRIVE_BASE}')
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print('Bukan di Colab — menggunakan path lokal.')

In [ ]:
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

ARTIFACTS_DIR = Path(f'{DRIVE_BASE}/artifacts_v2') if IN_COLAB else Path('artifacts_v2')
ARTIFACTS_DIR.mkdir(exist_ok=True)

MODEL_VERSION = '2.0.0-weighted-minmax'
print(f'Model: {MODEL_VERSION} | Seed: {RANDOM_STATE}')

## 1. Loading & Preprocessing Identik dengan v1

In [ ]:
DATA_PATH = f'{DRIVE_BASE}/datasets/TKPI_2017_dataset  .csv' if IN_COLAB else '../datasets/TKPI_2017_dataset  .csv'
df_raw = pd.read_csv(DATA_PATH)

NUMERIC_COLS = ['PROTEIN_g', 'ABU_g', 'KALSIUM_mg', 'BESI_mg', 'NATRIUM_mg',
                'TEMBAGA_mg', 'KARTOTAL_mcg', 'THIAMIN_mg', 'RIBOFLAVIN_mg',
                'NIASIN_mg', 'BDD_pct']
df = df_raw.copy()
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

COLUMN_MAP = {
    'KODE': 'food_code', 'NAMA_BAHAN': 'name', 'KATEGORI': 'category',
    'SUMBER': 'data_source', 'ENERGI_kal': 'energy_kcal', 'PROTEIN_g': 'protein_g',
    'LEMAK_g': 'fat_total_g', 'KH_g': 'carbs_g', 'SERAT_g': 'fiber_g',
    'KALSIUM_mg': 'calcium_mg', 'FOSFOR_mg': 'phosphorus_mg',
    'NATRIUM_mg': 'sodium_mg', 'KALIUM_mg': 'potassium_mg',
}
df = df.rename(columns=COLUMN_MAP)
print(f'Dataset: {len(df)} item')

In [ ]:
DASH_FEATURES = ['sodium_mg', 'potassium_mg', 'calcium_mg', 'fiber_g', 'fat_total_g']

NUTRIENT_WEIGHTS = {
    'sodium_mg':    {'direction': 'lower',  'weight': 0.30},
    'potassium_mg': {'direction': 'higher', 'weight': 0.25},
    'calcium_mg':   {'direction': 'higher', 'weight': 0.20},
    'fiber_g':      {'direction': 'higher', 'weight': 0.15},
    'fat_total_g':  {'direction': 'lower',  'weight': 0.10},
}

RELEVANT_CATEGORIES = ['Serealia', 'Umbi Berpati', 'Kacang & Biji', 'Sayuran',
                       'Buah', 'Daging & Unggas', 'Ikan, Kerang & Udang', 'Telur', 'Susu']

df_filtered = df[df['category'].isin(RELEVANT_CATEGORIES)].copy()
dash_non_null = df_filtered[DASH_FEATURES].notna().sum(axis=1)
df_filtered = df_filtered[dash_non_null >= 3].copy()

for feat in DASH_FEATURES:
    df_filtered[feat] = df_filtered.groupby('category')[feat].transform(lambda s: s.fillna(s.median()))
    df_filtered[feat] = df_filtered[feat].fillna(df_filtered[feat].median())
    df_filtered[feat] = df_filtered[feat].clip(lower=0)

df_clean = df_filtered.reset_index(drop=True)
print(f'Dataset bersih: {len(df_clean)} item')

## 2. Normalisasi dengan MinMaxScaler

**Perbedaan utama**: MinMaxScaler memetakan ke rentang [0,1], cocok untuk fitur skewed.

In [ ]:
X = df_clean[DASH_FEATURES].to_numpy(dtype=np.float64)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Bandingkan dengan StandardScaler (informational)
from sklearn.preprocessing import StandardScaler
ss = StandardScaler().fit(X)

print('=== PERBANDINGAN NORMALISASI ===')
for i, feat in enumerate(DASH_FEATURES):
    print(f'{feat:15s} | MinMax range: [{X_scaled[:,i].min():.3f}, {X_scaled[:,i].max():.3f}]')
    print(f'{"":15s} | StdScaler   : mean={ss.mean_[i]:.1f}, std={ss.scale_[i]:.1f}')

## 3. Weighted Cosine Similarity

Standard cosine memperlakukan semua fitur sama. Weighted cosine mengalikan vektor dengan akar bobot sebelum hitung similarity, sehingga fitur dengan bobot lebih tinggi (sodium 0.30) memberi pengaruh lebih besar.

$$\text{wcos}(a, b) = \frac{\sum_i w_i a_i b_i}{\sqrt{\sum_i w_i a_i^2} \sqrt{\sum_i w_i b_i^2}}$$

In [ ]:
def get_feature_weights():
    return np.array([NUTRIENT_WEIGHTS[f]['weight'] for f in DASH_FEATURES])

def weighted_cosine(user_vec_scaled, item_matrix_scaled, weights):
    """Cosine similarity dengan bobot per fitur."""
    sqrt_w = np.sqrt(weights)
    user_w = user_vec_scaled * sqrt_w
    items_w = item_matrix_scaled * sqrt_w
    return cosine_similarity(user_w.reshape(1, -1), items_w)[0]

weights = get_feature_weights()
print(f'Bobot fitur: {dict(zip(DASH_FEATURES, weights))}')

## 4. Demo Rekomendasi

In [ ]:
def calculate_personal_targets(profile):
    w, h, age, g = profile['weight_kg'], profile['height_cm'], profile['age'], profile['gender']
    bmr = (10*w) + (6.25*h) - (5*age) + (5 if g == 'laki-laki' else -161)
    targets = {
        'sodium_mg': 2300.0, 'potassium_mg': 4000.0,
        'calcium_mg': 1200.0 if age > 50 else 1000.0,
        'fiber_g': 38.0 if g == 'laki-laki' else 25.0,
        'fat_total_g': round(bmr * 0.27 / 9, 1),
    }
    if 'ckd' in profile.get('comorbidities', []):
        targets['sodium_mg'] = 1500.0
        targets['potassium_mg'] = 2000.0
    if profile.get('systolic_bp', 0) >= 150:
        targets['sodium_mg'] = 1500.0
    return targets

profile = {'gender': 'laki-laki', 'weight_kg': 70, 'height_cm': 170, 'age': 45,
           'comorbidities': [], 'systolic_bp': 140}
targets = calculate_personal_targets(profile)

user_vec = np.array([targets[f] for f in DASH_FEATURES])
user_vec_scaled = scaler.transform(user_vec.reshape(1, -1))[0]

sim = weighted_cosine(user_vec_scaled, X_scaled, weights)

df_clean['similarity'] = sim
top15 = df_clean.nlargest(15, 'similarity')[['food_code', 'name', 'category', 'similarity']]
print('=== TOP 15 (Weighted Cosine + MinMax) ===')
top15

## 5. Simpan Artefak v2

In [ ]:
joblib.dump(scaler, ARTIFACTS_DIR / 'scaler.pkl')
np.save(ARTIFACTS_DIR / 'item_matrix.npy', X_scaled)
np.save(ARTIFACTS_DIR / 'feature_weights.npy', weights)

with open(ARTIFACTS_DIR / 'food_ids.json', 'w', encoding='utf-8') as f:
    json.dump(df_clean['food_code'].tolist(), f, ensure_ascii=False)

metadata = {
    'version': MODEL_VERSION,
    'approach': 'Weighted Cosine Similarity + MinMaxScaler',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'random_state': RANDOM_STATE,
    'n_items': len(df_clean),
    'features': DASH_FEATURES,
    'feature_weights': weights.tolist(),
    'scaler_class': 'MinMaxScaler',
}
with open(ARTIFACTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

df_clean.to_csv(ARTIFACTS_DIR / 'food_items_clean.csv', index=False)

print('✓ Artefak v2 tersimpan di', ARTIFACTS_DIR.resolve())